# Gate 2 pilot: does plasticity loss show up on a drifting-dynamics stream?

**Purpose.** Before writing the proposal, find out cheaply whether a small online learner degrades across many shifts of a robot-arm-like dynamics stream, and whether the standard proxies track a direct probe of plasticity.

**What this notebook does**

1. Simulates a 3-joint servo arm whose physical parameters change between *regimes* (payload, friction, servo gain, damping, dead-zone, sensor bias). Abrupt, gradual, or mixed drift.
2. Trains a small MLP online to predict the next joint state from (state, command), one regime after another, with: plain Adam, L2, shrink-and-perturb, CBP, an *experimental* adaptive-rate CBP, and an oracle "reset at every shift" reference.
3. Adds a classical baseline: forgetting-factor recursive least squares (RLS).
4. Measures a **probe-based plasticity score** (copy the net, train K steps on a held-out probe regime, compare to a fresh net) next to common proxies (dormant fraction, effective rank, weight magnitude, gradient norm).
5. Prints a **Gate 2 verdict** and a proxy-vs-probe correlation table.
6. Has a loader (`stream_from_logs`) so the same pipeline runs on real logged arm data.

**Read before trusting anything**

- The plant is synthetic. The RLS baseline is *given the right feature structure* (it knows about gravity and friction terms), so it is an optimistic competitor. It does not know about the dead-zone, which is the "unmodelled effect" a neural net could add.
- CBP here is my re-implementation of the adaptive-contribution-utility version. Compare it against the authors' reference code before you claim a reproduction.
- The adaptive-rate controller is a naive starting point (proportional to the dormant fraction), not a result.
- Start with `QUICK = True` to shake out bugs, then set it to `False`.
- Scores: `gap = end_loss / end_loss_of_reset_baseline` on the same regime. `gap < 1` means better than retraining from scratch at every shift. `probe` score > 1 means the net learns the probe task faster than a fresh one.

In [ ]:
import copy, math, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)     # tiny MLPs: CPU, single thread is fine (and lets you run several notebooks in parallel)

QUICK = True                 # True: short smoke test. False: full pilot.

CFG = dict(
    # --- stream ---
    n_joints=3,
    n_regimes=60 if QUICK else 400,
    drift_mode="abrupt",     # "abrupt" | "gradual" | "mixed"
    gradual_step=0.15,       # random-walk step in normalised parameter space
    T=2500, T_train=2000,    # samples simulated per regime (first T_train train, rest test)
    n_probe_tasks=2,
    # --- learner ---
    width=128, depth=3, act="relu",          # "relu" | "tanh" | "leaky_relu"
    lr=1e-3, beta1=0.9, beta2=0.999,
    steps_per_regime=200, batch=32,
    # --- CBP ---
    cbp_rho=1e-4, cbp_maturity=100, cbp_decay=0.99,
    adam_state_on_reset="zero",              # "zero" | "keep"  (sub-ablation: Adam moments of reset units)
    adapt_rho_max=1e-3, adapt_target=0.10,   # adaptive CBP: rho = rho_max * min(1, dormant_ema / target)
    # --- other baselines ---
    l2_wd=1e-4, sp_shrink=0.9999, sp_noise=1e-4,
    # --- probe ---
    probe_every=5 if QUICK else 10, probe_K=50, n_fresh_ref=3,
    # --- experiment ---
    seeds=[0, 1] if QUICK else [0, 1, 2, 3, 4],
)
METHODS = ["none", "l2", "sp", "cbp", "cbp_adaptive", "reset"]   # "reset" is the oracle reference, keep it

## 1. Drifting servo-arm simulator

Each joint is an independent servo: a position command `u`, an inner loop that pulls the joint toward `u`, gravity from a payload, viscous and Coulomb-like friction, and a dead-zone in the command error. The *measured* angle also carries a per-regime offset (sensor bias). The model never sees the physical parameters, only `[q, qd, u]` in and `[Δq, Δqd]` out, so it works directly in joint space and needs no kinematics.

In [ ]:
PARAMS = {                      # name: (low, high)
    "mass":     (0.0, 2.0),     # payload -> gravity torque
    "fric":     (0.0, 1.0),     # Coulomb-like friction
    "gain":     (8.0, 30.0),    # servo stiffness
    "damp":     (1.0, 6.0),     # viscous damping
    "deadzone": (0.0, 0.08),    # dead-zone on command error (NOT in the RLS features)
    "bias":     (-0.1, 0.1),    # sensor offset on measured angle
}
PNAMES = list(PARAMS)
G, DT = 3.0, 0.02

def sample_unit(R, J, mode, rng, step):
    """Normalised parameters in [0,1], shape (R, J, n_params)."""
    u = rng.uniform(size=(R, J, len(PNAMES)))
    if mode == "abrupt":
        return u
    reflect = lambda x: 1 - np.abs(1 - np.abs(x))
    for r in range(1, R):
        jump = (mode == "mixed") and rng.uniform() < 0.1
        if not jump:
            u[r] = reflect(u[r - 1] + step * rng.normal(size=(J, len(PNAMES))))
    return u

def unit_to_params(u):
    return {n: PARAMS[n][0] + u[..., i] * (PARAMS[n][1] - PARAMS[n][0]) for i, n in enumerate(PNAMES)}

def simulate(p, T, rng):
    """Roll out all regimes in parallel. Returns X (R,T,3J) = [q_obs, qd_obs, u] and Y (R,T,2J) = [dq_obs, dqd_obs]."""
    R, J = p["mass"].shape
    q = rng.uniform(-0.5, 0.5, (R, J)); qd = np.zeros((R, J))
    amp = rng.uniform(0.3, 1.0, (R, J, 2)); w = rng.uniform(0.5, 3.0, (R, J, 2)); ph = rng.uniform(0, 2 * np.pi, (R, J, 2))
    oq = np.empty((T + 1, R, J)); oqd = np.empty((T + 1, R, J)); U = np.empty((T, R, J))
    def measure():
        return q + p["bias"] + 0.002 * rng.normal(size=q.shape), qd + 0.01 * rng.normal(size=qd.shape)
    oq[0], oqd[0] = measure()
    for t in range(T):
        u = (amp * np.sin(w * (t * DT) + ph)).sum(-1)
        e = u - q
        e_eff = e - np.clip(e, -p["deadzone"], p["deadzone"])
        acc = p["gain"] * e_eff - p["damp"] * qd - p["mass"] * G * np.cos(q) - p["fric"] * np.tanh(qd / 0.05)
        qd = qd + DT * acc
        q = q + DT * qd
        U[t] = u
        oq[t + 1], oqd[t + 1] = measure()
    X = np.concatenate([oq[:-1], oqd[:-1], U], -1)
    Y = np.concatenate([oq[1:] - oq[:-1], oqd[1:] - oqd[:-1]], -1)
    return X.transpose(1, 0, 2), Y.transpose(1, 0, 2)

def make_stream(cfg, seed):
    rng = np.random.default_rng(1000 + seed)
    R, J = cfg["n_regimes"], cfg["n_joints"]
    X, Y = simulate(unit_to_params(sample_unit(R, J, cfg["drift_mode"], rng, cfg["gradual_step"])), cfg["T"], rng)
    Xp, Yp = simulate(unit_to_params(sample_unit(cfg["n_probe_tasks"], J, "abrupt", rng, 0)), cfg["T"], rng)
    return dict(X=X, Y=Y, Xp=Xp, Yp=Yp, J=J)

def prepare(stream):
    """Standardise with stream-wide statistics and convert to tensors. Works for synthetic or real streams."""
    X, Y = stream["X"], stream["Y"]
    D, O = X.shape[-1], Y.shape[-1]
    xm, xs = X.reshape(-1, D).mean(0), X.reshape(-1, D).std(0) + 1e-8
    ym, ys = Y.reshape(-1, O).mean(0), Y.reshape(-1, O).std(0) + 1e-8
    f = lambda a, m, s: torch.tensor((a - m) / s, dtype=torch.float32)
    return dict(Xn=f(X, xm, xs), Yn=f(Y, ym, ys), Xpn=f(stream["Xp"], xm, xs), Ypn=f(stream["Yp"], ym, ys), ys=ys, D=D, O=O)

## 2. Model, CBP, proxies, probe

**CBP** follows the continual-backprop recipe: per hidden unit keep an age and an *adaptive contribution utility* (`|h - h_bar| * sum|w_out| / sum|w_in|`, exponentially averaged). Each step, a fractional counter `rho * n_mature_units` accumulates; whenever it passes 1, the lowest-utility *mature* unit(s) get fresh incoming weights, zero outgoing weights and zero bias, and their utility and age reset.

`adam_state_on_reset` controls the sub-ablation I suggested earlier: zero the Adam moments of the reset weights, or keep the stale ones.

**Probe.** Deep-copy the network, give the copy a *fresh optimiser*, train `probe_K` steps on a held-out probe regime, and divide the loss that freshly initialised networks reach (averaged over `n_fresh_ref` inits) by the copy's loss. Score `1.0` ≈ as plastic as a fresh net, `<1` degraded. The probe is only an operational definition: it depends on the probe tasks, `K`, the optimiser, and the learning rate.

In [ ]:
ACTS = {"relu": torch.relu, "tanh": torch.tanh, "leaky_relu": F.leaky_relu}

def init_linear(layer, act):
    nn.init.kaiming_uniform_(layer.weight, nonlinearity="tanh" if act == "tanh" else "relu")
    nn.init.zeros_(layer.bias)

class MLP(nn.Module):
    def __init__(self, din, dout, width, depth, act):
        super().__init__()
        dims = [din] + [width] * depth + [dout]
        self.layers = nn.ModuleList([nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.act_name, self.act = act, ACTS[act]
        self.reset_all()
    def reset_all(self):
        for l in self.layers:
            init_linear(l, self.act_name)
    def forward(self, x, return_acts=False):
        acts = []
        for l in self.layers[:-1]:
            x = self.act(l(x))
            acts.append(x)
        out = self.layers[-1](x)
        return (out, acts) if return_acts else out

def make_opt(net, cfg, wd=0.0):
    return torch.optim.Adam(net.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]), weight_decay=wd)

@torch.no_grad()
def eval_loss(net, X, Y):
    return F.mse_loss(net(X), Y).item()

def dormant_fraction(acts, tau=0.025):
    """ReDo-style dormant fraction: unit's mean |activation| relative to its layer's average, averaged over layers."""
    fr = []
    for h in acts:
        s = h.detach().abs().mean(0)
        s = s / (s.mean() + 1e-8)
        fr.append((s <= tau).float().mean().item())
    return float(np.mean(fr))

class CBP:
    def __init__(self, net, opt, rho, maturity, decay, adam_state="zero"):
        self.net, self.opt = net, opt
        self.rho, self.maturity, self.decay, self.adam_state = rho, maturity, decay, adam_state
        H = [l.out_features for l in net.layers[:-1]]
        self.util = [torch.zeros(h) for h in H]
        self.age = [torch.zeros(h) for h in H]
        self.hbar = [torch.zeros(h) for h in H]
        self.counter = [0.0] * len(H)
        self.n_replaced = 0

    @torch.no_grad()
    def step(self, acts, rho=None):
        rho = self.rho if rho is None else rho
        for i, h in enumerate(acts):
            h = h.detach()
            self.age[i] += 1
            self.hbar[i] = self.decay * self.hbar[i] + (1 - self.decay) * h.mean(0)
            w_in, w_out = self.net.layers[i].weight, self.net.layers[i + 1].weight
            contrib = (h - self.hbar[i]).abs().mean(0) * w_out.abs().sum(0) / (w_in.abs().sum(1) + 1e-8)
            self.util[i] = self.decay * self.util[i] + (1 - self.decay) * contrib
            elig = (self.age[i] > self.maturity).nonzero().squeeze(1)
            if elig.numel() == 0:
                continue
            self.counter[i] += rho * elig.numel()
            n_rep = int(self.counter[i])
            if n_rep < 1:
                continue
            self.counter[i] -= n_rep
            n_rep = min(n_rep, elig.numel())
            idx = elig[torch.argsort(self.util[i][elig])[:n_rep]]
            self._reinit(i, idx)

    @torch.no_grad()
    def _reinit(self, i, idx):
        l_in, l_out = self.net.layers[i], self.net.layers[i + 1]
        fresh = torch.empty(len(idx), l_in.in_features)
        nn.init.kaiming_uniform_(fresh, nonlinearity="tanh" if self.net.act_name == "tanh" else "relu")
        l_in.weight[idx] = fresh
        l_in.bias[idx] = 0.0
        l_out.weight[:, idx] = 0.0
        self.util[i][idx] = 0.0
        self.age[i][idx] = 0.0
        self.hbar[i][idx] = 0.0
        self.n_replaced += len(idx)
        if self.adam_state == "zero":
            for p, sl in ((l_in.weight, (idx,)), (l_in.bias, (idx,)), (l_out.weight, (slice(None), idx))):
                st = self.opt.state.get(p, {})
                for k in ("exp_avg", "exp_avg_sq"):
                    if k in st:
                        st[k][sl] = 0.0

def proxies(net, xb, yb):
    """Cheap proxies on a fixed batch."""
    with torch.no_grad():
        _, acts = net(xb, True)
        dorm = dormant_fraction(acts)
        s = torch.linalg.svdvals(acts[-1])
        cs = torch.cumsum(s, 0) / s.sum()
        srank = (int((cs < 0.99).sum().item()) + 1) / acts[-1].shape[1]      # effective rank, delta = 0.01
        wmag = float(np.mean([l.weight.abs().mean().item() for l in net.layers]))
    net.zero_grad(set_to_none=True)
    F.mse_loss(net(xb), yb).backward()
    gn = math.sqrt(sum((p.grad ** 2).sum().item() for p in net.parameters()))
    net.zero_grad(set_to_none=True)
    return dict(dormant=dorm, srank_frac=srank, wmag=wmag, gradnorm=gn)

def train_steps(net, opt, X, Y, steps, batch, gen):
    for _ in range(steps):
        idx = torch.randint(0, X.shape[0], (batch,), generator=gen)
        loss = F.mse_loss(net(X[idx]), Y[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

class Probe:
    """Probe-based plasticity score: loss_fresh / loss_copy after K steps on held-out probe regimes."""
    def __init__(self, cfg, data, seed):
        self.cfg, self.seed = cfg, seed
        Ttr = cfg["T_train"]
        self.tasks = [(data["Xpn"][p, :Ttr], data["Ypn"][p, :Ttr], data["Xpn"][p, Ttr:], data["Ypn"][p, Ttr:])
                      for p in range(data["Xpn"].shape[0])]
        torch.manual_seed(seed + 999)
        self.fresh = []
        for (Xtr, Ytr, Xte, Yte) in self.tasks:
            ls = []
            for k in range(cfg["n_fresh_ref"]):
                n = MLP(data["D"], data["O"], cfg["width"], cfg["depth"], cfg["act"])
                train_steps(n, make_opt(n, cfg), Xtr, Ytr, cfg["probe_K"], cfg["batch"], torch.Generator().manual_seed(seed + k))
                ls.append(eval_loss(n, Xte, Yte))
            self.fresh.append(float(np.mean(ls)))

    def score(self, net):
        ratios = []
        for (Xtr, Ytr, Xte, Yte), ref in zip(self.tasks, self.fresh):
            c = copy.deepcopy(net)
            train_steps(c, make_opt(c, self.cfg), Xtr, Ytr, self.cfg["probe_K"], self.cfg["batch"], torch.Generator().manual_seed(self.seed))
            ratios.append(ref / eval_loss(c, Xte, Yte))
        return float(np.mean(ratios))

## 3. Online learner over the stream

Per regime: `steps_per_regime` minibatch updates drawn from that regime's training pool. Logged per regime: loss on the regime's held-out test split **before** training (`start`, the adaptation shock), the loss on it **after** (`end`), and the mean training loss over the first 20 steps (`early`). Every `probe_every` regimes the proxies and the probe score are recorded.

Methods:
- `none`: plain Adam. `l2`: Adam with coupled weight decay. `sp`: shrink-and-perturb every step.
- `cbp`: fixed replacement rate. `cbp_adaptive`: rate driven by the dormant-fraction EMA (experimental).
- `reset`: oracle that re-initialises net and optimiser at *every* shift. This is your "might as well retrain" reference and the denominator of `gap`.

In [ ]:
def run_method(method, cfg, data, probe, seed):
    Xn, Yn = data["Xn"], data["Yn"]
    R, Ttr, S, B = Xn.shape[0], cfg["T_train"], cfg["steps_per_regime"], cfg["batch"]
    torch.manual_seed(seed * 100 + 7)                       # same init for every method within a seed
    net = MLP(data["D"], data["O"], cfg["width"], cfg["depth"], cfg["act"])
    wd = cfg["l2_wd"] if method == "l2" else 0.0
    opt = make_opt(net, cfg, wd)
    cbp = CBP(net, opt, cfg["cbp_rho"], cfg["cbp_maturity"], cfg["cbp_decay"], cfg["adam_state_on_reset"]) \
          if method in ("cbp", "cbp_adaptive") else None
    gen = torch.Generator().manual_seed(seed)
    rows, prox_rows, dorm_ema = [], [], 0.0
    for r in range(R):
        if method == "reset":
            net.reset_all()
            opt = make_opt(net, cfg, wd)
        Xtr, Ytr, Xte, Yte = Xn[r, :Ttr], Yn[r, :Ttr], Xn[r, Ttr:], Yn[r, Ttr:]
        start = eval_loss(net, Xte, Yte)
        early = []
        for s in range(S):
            idx = torch.randint(0, Ttr, (B,), generator=gen)
            out, acts = net(Xtr[idx], True)
            loss = F.mse_loss(out, Ytr[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            if s < 20:
                early.append(loss.item())
            if method == "sp":
                with torch.no_grad():
                    for p in net.parameters():
                        p.mul_(cfg["sp_shrink"]).add_(torch.randn_like(p), alpha=cfg["sp_noise"])
            if cbp is not None:
                rho = cfg["cbp_rho"]
                if method == "cbp_adaptive":
                    dorm_ema = 0.99 * dorm_ema + 0.01 * dormant_fraction(acts)
                    rho = cfg["adapt_rho_max"] * min(1.0, dorm_ema / cfg["adapt_target"])
                cbp.step(acts, rho)
        rows.append(dict(method=method, seed=seed, regime=r, start=start, early=float(np.mean(early)),
                         end=eval_loss(net, Xte, Yte), replaced=cbp.n_replaced if cbp else 0))
        if r % cfg["probe_every"] == 0 or r == R - 1:
            pr = proxies(net, Xte, Yte)
            pr.update(method=method, seed=seed, regime=r, probe=probe.score(net))
            prox_rows.append(pr)
    return rows, prox_rows

## 4. Classical baseline: forgetting-factor RLS

Per joint, a linear model in the features `[u - q, qd, cos q, tanh(qd/0.05), 1]` predicting `(Δq, Δqd)`. These are the right physical terms for this plant, so this is a deliberately strong, optimistic baseline. It cannot represent the dead-zone or the sensor-bias effect on `cos q`. It sees the same minibatch stream as the networks, with exponential forgetting per batch (`lam`), and the loss is reported in the same standardised units.

In [ ]:
def rls_features(X, J):
    q, qd, u = X[..., :J], X[..., J:2 * J], X[..., 2 * J:]
    return np.stack([u - q, qd, np.cos(q), np.tanh(qd / 0.05), np.ones_like(q)], -1)    # (..., J, 5)

def run_rls(stream, ys, cfg, seed, lam=0.98, ridge=1e-3):
    X, Y, J = stream["X"], stream["Y"], stream["J"]
    R, Ttr, S, B = X.shape[0], cfg["T_train"], cfg["steps_per_regime"], cfg["batch"]
    Yj = np.stack([Y[..., :J], Y[..., J:]], -1)                  # (R, T, J, 2)
    ysj = np.stack([ys[:J], ys[J:]], -1)                         # (J, 2)
    Fe = rls_features(X, J)                                      # (R, T, J, 5)
    nf = Fe.shape[-1]
    A, b = np.zeros((J, nf, nf)), np.zeros((J, nf, 2))
    rng = np.random.default_rng(seed + 5)
    def test_loss(r):
        theta = np.linalg.solve(A + ridge * np.eye(nf), b)       # (J, nf, 2)
        pred = np.einsum("tjf,jfk->tjk", Fe[r, Ttr:], theta)
        return float((((pred - Yj[r, Ttr:]) / ysj) ** 2).mean())
    rows = []
    for r in range(R):
        start = test_loss(r)
        for s in range(S):
            idx = rng.integers(0, Ttr, B)
            Ph, Yb = Fe[r, idx], Yj[r, idx]
            A = lam * A + np.einsum("bjf,bjg->jfg", Ph, Ph)
            b = lam * b + np.einsum("bjf,bjk->jfk", Ph, Yb)
        rows.append(dict(method="rls", seed=seed, regime=r, start=start, early=np.nan, end=test_loss(r), replaced=0))
    return rows

In [ ]:
all_res, all_prox = [], []
t0 = time.time()
for seed in CFG["seeds"]:
    stream = make_stream(CFG, seed)
    data = prepare(stream)
    probe = Probe(CFG, data, seed)
    print(f"seed {seed}: probe fresh-reference losses = {[round(x, 4) for x in probe.fresh]}")
    for m in METHODS:
        t1 = time.time()
        r, p = run_method(m, CFG, data, probe, seed)
        all_res += r; all_prox += p
        print(f"  {m:13s} {time.time() - t1:6.1f}s  mean end-loss (last 10 regimes) {np.mean([x['end'] for x in r[-10:]]):.4f}"
              f"   units replaced {r[-1]['replaced']}")
    all_res += run_rls(stream, data["ys"], CFG, seed)
    pd.DataFrame(all_res).to_csv("pilot_results.csv", index=False)
    pd.DataFrame(all_prox).to_csv("pilot_proxies.csv", index=False)
print(f"done in {(time.time() - t0) / 60:.1f} min -> pilot_results.csv, pilot_proxies.csv")

## 5. Results

Re-runnable from the CSVs, so you can restart the kernel after a long run.

In [ ]:
res = pd.read_csv("pilot_results.csv")
prox = pd.read_csv("pilot_proxies.csv")

ref = res[res.method == "reset"][["seed", "regime", "end"]].rename(columns={"end": "ref_end"})
res = res.merge(ref, on=["seed", "regime"])
res["gap"] = res["end"] / res["ref_end"]            # < 1: better than retraining from scratch at this shift
R = int(res.regime.max()) + 1
win = max(5, R // 20)

def smooth(df, col, w):
    d = df.sort_values("regime").copy()
    d[col + "_s"] = d.groupby(["method", "seed"])[col].transform(lambda s: s.rolling(w, min_periods=1).mean())
    return d.groupby(["method", "regime"])[col + "_s"].agg(["mean", "std"]).fillna(0).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, col, title in zip(axes, ["end", "gap", "start"],
                          ["loss after training on each regime", "gap vs reset baseline (<1 beats retraining)", "loss at the moment of each shift"]):
    g = smooth(res, col, win)
    for m, d in g.groupby("method"):
        ax.plot(d.regime, d["mean"], label=m, lw=2 if m in ("none", "cbp", "rls") else 1.2)
        ax.fill_between(d.regime, d["mean"] - d["std"], d["mean"] + d["std"], alpha=0.12)
    ax.set_title(title); ax.set_xlabel("regime"); ax.set_yscale("log" if col != "gap" else "linear")
axes[1].axhline(1, color="k", ls=":", lw=1); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Gate 2 verdict

For each method: mean `gap` in an early window (after a short warm-up) vs the last window. `late/early > 1` means that method learns each new regime *relatively worse* than a fresh network as the stream goes on. The thresholds below are arbitrary and deliberately transparent, so change them and see.

In [ ]:
THRESH = 1.10
warm = min(5, R // 10); w = max(5, R // 5)
def wmean(lo, hi):
    return res[(res.regime >= lo) & (res.regime < hi)].groupby(["method", "seed"])["gap"].mean()
early, late = wmean(warm, warm + w), wmean(R - w, R)
ratio = (late / early).rename("late_over_early").reset_index()
summary = ratio.groupby("method")["late_over_early"].agg(["mean", "min", "max"])
summary["late_gap_vs_reset"] = late.groupby("method").mean()
print(f"early window = regimes [{warm}, {warm + w}), late window = [{R - w}, {R}), seeds = {[int(x) for x in sorted(res.seed.unique())]}\n")
print(summary.round(3).to_string())

def verdict(m):
    r = ratio[ratio.method == m]["late_over_early"]
    if len(r) == 0: return "n/a"
    if r.min() > THRESH: return "PLASTICITY LOSS VISIBLE (every seed)"
    if r.mean() > THRESH: return "AMBIGUOUS (mean above threshold, some seeds not)"
    return "NOT DETECTED at this scale"
print("\nGate 2 verdict for the plain online learner ('none'):", verdict("none"))
print("If NOT DETECTED: raise lr, steps_per_regime or n_regimes, try width=64, beta2=0.9, drift_mode='mixed', then rerun.")
print("If a method has late_gap_vs_reset < 1, it beats retraining at every shift; RLS < 1 means the classical baseline wins.")

### Do the proxies track the probe, and does the probe warn before adaptation gets worse?

- `spearman vs probe`: proxy against the probe score.
- `spearman vs future_gap`: proxy against the mean `gap` over the *next* `W` regimes. A useful plasticity monitor should be **negatively** correlated with `future_gap` for the probe (higher plasticity, lower gap) and show the right sign for a proxy.
- "pooled" mixes methods (dominated by between-method differences). "within" averages the per-method correlations (dominated by drift over time). Both matter, and they can disagree.
- `reset` is excluded because it resets every shift.

In [ ]:
W = 5
gaps = {k: g.sort_values("regime").set_index("regime")["gap"] for k, g in res.groupby(["method", "seed"])}
def fut(row):
    v = gaps[(row["method"], row["seed"])].loc[row["regime"] + 1: row["regime"] + W]
    return v.mean() if len(v) else np.nan
prox["future_gap"] = prox.apply(fut, axis=1)
p2 = prox[prox.method != "reset"].copy()
cols = ["dormant", "srank_frac", "wmag", "gradnorm", "probe"]
spear = lambda a, b: a.rank().corr(b.rank())        # Spearman = Pearson on ranks (no scipy needed)

def within(target):
    out = {}
    for c in cols:
        cs = [spear(d[c], d[target]) for _, d in p2.groupby("method") if d[c].nunique() > 1 and d[target].nunique() > 1]
        out[c] = float(np.nanmean(cs)) if cs else np.nan
    return out

tab = pd.DataFrame({
    "pooled vs probe":      {c: spear(p2[c], p2["probe"]) for c in cols},
    "pooled vs future_gap": {c: spear(p2[c], p2["future_gap"]) for c in cols},
    "within vs probe":      within("probe"),
    "within vs future_gap": within("future_gap"),
})
print(f"n checkpoints = {len(p2)}, W = {W}")
print(tab.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m, d in p2.groupby("method"):
    axes[0].scatter(d["probe"], d["future_gap"], label=m, s=14, alpha=0.7)
axes[0].set_xlabel("probe score (higher = more plastic)"); axes[0].set_ylabel(f"future gap (next {W} regimes)"); axes[0].legend(fontsize=8)
for m, d in p2.groupby("method"):
    g = d.groupby("regime")["probe"].mean()
    axes[1].plot(g.index, g.values, marker="o", ms=3, label=m)
axes[1].axhline(1, color="k", ls=":", lw=1); axes[1].set_xlabel("regime"); axes[1].set_ylabel("probe score")
plt.tight_layout(); plt.show()

## 6. Plug in real logged arm data

The pipeline only needs `X (R, T, 3J)`, `Y (R, T, 2J)` and held-out probe windows `Xp, Yp`. `stream_from_logs` builds those from a dataframe of smart-servo logs. Expected columns: `q0..q{J-1}` (measured angle), `qd0..` (measured velocity), `u0..` (commanded angle), and `condition` (e.g. `no_load`, `load_200g`, `fast`), rows in time order within each condition. Each regime is a random contiguous window of `T` samples from a random condition. Probe windows come from the last 20% of each recording in time, training windows from the first 80%, so each condition needs at least `5 * T` samples (12,500 samples, about 4 minutes at 50 Hz, with the default `T`).

Caveats: with only a handful of conditions, regimes **recur**. That does not add forgetting to the *measurement* (you only score learning on the current regime), but it does let the network warm-start from earlier regimes. Add software perturbations (command delay, gain scaling, bias) to your recordings to widen the pool of distinct regimes.

In [ ]:
def stream_from_logs(df, J, cfg, seed=0, cond_col="condition"):
    rng = np.random.default_rng(seed)
    T = cfg["T"]
    qc, qdc, uc = [[f"{p}{j}" for j in range(J)] for p in ("q", "qd", "u")]
    train, hold = [], []
    for c, g in df.groupby(cond_col, sort=False):
        obs, U = g[qc + qdc].to_numpy(float), g[uc].to_numpy(float)
        X = np.concatenate([obs[:-1], U[:-1]], 1)
        Y = obs[1:] - obs[:-1]
        cut = int(0.8 * len(X))
        if cut >= T:           train.append((X[:cut], Y[:cut]))
        if len(X) - cut >= T:  hold.append((X[cut:], Y[cut:]))
    assert train and hold, f"each condition needs >= {5 * T} samples (T={T}; the last 20% is held out for probe windows)"
    def windows(pool, n):
        Xs, Ys = [], []
        for _ in range(n):
            X, Y = pool[rng.integers(len(pool))]
            s = rng.integers(0, len(X) - T + 1)
            Xs.append(X[s:s + T]); Ys.append(Y[s:s + T])
        return np.stack(Xs), np.stack(Ys)
    X, Y = windows(train, cfg["n_regimes"])
    Xp, Yp = windows(hold, cfg["n_probe_tasks"])
    return dict(X=X, Y=Y, Xp=Xp, Yp=Yp, J=J)

# Example (uncomment and point at your logs). Then rerun section 3 with `stream = stream_from_logs(...)`
# in place of `make_stream(CFG, seed)`, and drop "rls" if you do not want the classical baseline there.
# logs = pd.read_csv("arm_logs.csv")
# stream = stream_from_logs(logs, J=6, cfg=CFG, seed=0)

## What to do with the outcome

| What you see | What it means for the proposal |
|---|---|
| `none` shows `late/early > 1.1` in every seed, CBP/adaptive CBP reduce it | Lead with the application confidently: plasticity loss appears on drifting dynamics streams and mitigation helps. This is your preliminary result. |
| `none` shows no degradation | Raise difficulty (lr, steps, regimes, narrower net, `beta2=0.9`, `drift_mode="mixed"`). If it still does not appear, lead with the monitor/proxy study and present the arm as a feasibility study with a stated boundary. |
| Degradation appears but CBP does not help | Check `replaced` counts, `cbp_rho`, `cbp_maturity`, and `adam_state_on_reset` (`zero` vs `keep`) before concluding anything. Compare against the authors' reference code. |
| `rls` has `late_gap_vs_reset` far below the networks | Expected on a clean plant. The honest pitch is the neural *residual* on top of a classical model, with the dead-zone as the unmodelled part. |
| Probe tracks `future_gap` where proxies do not | That is the metric contribution. Check it across several `probe_K` values and probe tasks before believing it. |